# 📄 Banco de Dados de Documentos (NoSQL)
> **Missão:** Entender a diferença entre recuperar um documento atrelado a um ID versus fazer o banco "ler" todos os JSONs para buscar um dado aninhado.
> **Arquivo:** Os dados estão armazenados localmente no arquivo `catalogo_produtos.json`.

Neste laboratório, temos um catálogo onde cada produto tem atributos diferentes armazenados no campo `especificacoes`. Vamos comparar a performance de uma busca direta contra uma busca profunda não-indexada.

In [1]:
import json
import os
import random

file_path = 'catalogo_produtos.json'

if not os.path.exists(file_path):
    print("⏳ Gerando 'catalogo_produtos.json' no disco (isso pode levar alguns segundos)...")
    
    # Gerando 100.000 documentos com estruturas aninhadas
    dados = {}
    for i in range(1, 100001):
        dados[f"doc_{i}"] = {
            "_id": f"doc_{i}",
            "nome": f"Produto_{i}",
            "categoria": random.choice(["Eletrônicos", "Livros", "Roupas"]),
            "especificacoes": {
                "cor": random.choice(["preto", "branco", "azul", "vermelho"]),
                "peso_gramas": random.randint(100, 2000)
            }
        }
        
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(dados, f)
        
    print("✅ Arquivo 'catalogo_produtos.json' criado e salvo com sucesso na pasta!")
else:
    print("✅ Arquivo 'catalogo_produtos.json' já existe. Pronto para as consultas!")

✅ Arquivo 'catalogo_produtos.json' já existe. Pronto para as consultas!


In [2]:
import time
import json
import ipywidgets as widgets
from ipywidgets import interact

# 1. Carregamento dos dados (Simulando o banco carregando a coleção)
with open('catalogo_produtos.json', 'r', encoding='utf-8') as f:
    colecao = json.load(f)

# 2. Função de Visualização
def exibir_resultado(query: str, titulo: str, explicacao: str, tempo_ms: float, docs_lidos: int, resultado):
    print(f"\n{'=' * 80}")
    print("⚙️  PAINEL DE TESTES NOSQL (DOCUMENTOS)")
    print(f"{'=' * 80}")

    print("\n💻 QUERY EXECUTADA")
    print(f"{'-' * 80}")
    print(query.strip())

    print("\n📊 RESUMO")
    print(f"{'-' * 80}")
    print(f"Tempo de execução : {tempo_ms:.2f} ms")
    print(f"Documentos lidos  : {docs_lidos:,}".replace(",", "."))
    print(f"Resultado         : {titulo}")

    print("\n📝 O QUE ACONTECEU?")
    print(f"{'-' * 80}")
    print(explicacao)

    print("\n📋 RESULTADO DA CONSULTA (Amostra do JSON)")
    print(f"{'-' * 80}")
    
    if not resultado:
        print("Nenhum documento encontrado.")
    else:
        # Pega apenas o primeiro item para não inundar a tela com milhares de linhas
        amostra = resultado[0] if isinstance(resultado, list) else resultado
        print(json.dumps(amostra, indent=2, ensure_ascii=False))

    print(f"\n{'=' * 80}")


# 3. Lógica de Execução
def testar_documentos(tipo_query: str):
    inicio = time.time()

    if tipo_query == "Eficiente (Busca direta por _id)":
        query = """
db.catalogo.findOne({ "_id": "doc_85000" })
"""
        # Acesso direto à chave primária do dicionário (O(1))
        resultado = colecao.get("doc_85000")
        
        titulo = "🚀 Busca Rápida por ID"
        explicacao = (
            "Os bancos de documentos guardam o JSON atrelado a um ID fixo.\n"
            "Buscar pelo ID é imediato, pois o banco sabe exatamente qual "
            "gaveta abrir no disco, sem precisar interpretar o texto dentro."
        )
        docs_lidos = 1
        
    else:
        query = """
db.catalogo.find({ "especificacoes.peso_gramas": 1500 })
"""
        # Varredura O(N) buscando em propriedades aninhadas
        resultado = [doc for doc in colecao.values() if doc["especificacoes"]["peso_gramas"] == 1500]
        
        titulo = "🚨 Escaneamento Profundo (Deep Scan)"
        explicacao = (
            "A propriedade 'peso_gramas' está aninhada e não possui índice.\n"
            "O banco foi forçado a abrir todos os 100.000 documentos, decodificar "
            "o JSON de cada um e verificar a regra internamente. Isso esmaga a CPU."
        )
        docs_lidos = 100_000

    fim = time.time()
    tempo_ms = (fim - inicio) * 1000

    exibir_resultado(query, titulo, explicacao, tempo_ms, docs_lidos, resultado)


# 4. Interface Gráfica
opcoes_cenarios = [
    "Eficiente (Busca direta por _id)",
    "Ineficiente (Busca em campo aninhado sem índice)"
]

interact(
    testar_documentos,
    tipo_query=widgets.RadioButtons(
        options=opcoes_cenarios,
        description="Cenário:",
        layout={'width': 'max-content'}
    )
)

interactive(children=(RadioButtons(description='Cenário:', layout=Layout(width='max-content'), options=('Efici…

<function __main__.testar_documentos(tipo_query: str)>